# RDKV on Google Colab (A100): Team 6, v4
Same harness and settings as the baseline: bf16, one GPU, 800 records, no `--max-prompt-tokens`.

**Before running:**
1. **Runtime → Change runtime type → A100 GPU** (needs Colab Pro / compute units).
2. 🔑 Secrets (left bar): add `HF_TOKEN` = your Hugging Face token, and turn **Notebook access** on.
3. 📁 Files (left bar): upload **rdkv_kaggle_v4.zip** into `/content`. Don't unzip it.

Results are written **straight to Google Drive** (`MyDrive/rdkv_results`). If Colab disconnects, run all cells again and it **continues where it stopped**.
Keep this tab open and the laptop plugged in and awake.

In [ ]:
# 0. CONFIG: ratios to run in this notebook, in order
RATIOS = ["0.8", "0.95"]

In [ ]:
# 1. GPU (expect A100)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Drive, token, repo, RDKV files
import os, sys, glob, zipfile, shutil, subprocess
from google.colab import drive, userdata
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF token found:", bool(os.environ["HF_TOKEN"]))

REPO = "/content/repo"
if not os.path.exists(REPO + "/harness/run_longbench.py"):
    subprocess.run(["git", "clone", "-q", "https://github.com/samruddhichitnis02/298a-kv-cache-compression.git", REPO], check=True)
z = [p for p in glob.glob("/content/*.zip") + glob.glob("/content/drive/MyDrive/**/rdkv_kaggle_v4.zip", recursive=True)
     if "members/manjot/V4" in zipfile.ZipFile(p).namelist()]
assert z, "upload rdkv_kaggle_v4.zip to /content (Files panel on the left)"
zipfile.ZipFile(z[0]).extractall("/content/rdkv_v4")
shutil.copytree("/content/rdkv_v4/members/manjot", REPO + "/members/manjot", dirs_exist_ok=True)
shutil.copy("/content/rdkv_v4/harness/run_longbench.py", REPO + "/harness/run_longbench.py")

# results live on Drive so a disconnect loses nothing (the harness resumes from what is there)
OUT = "/content/drive/MyDrive/rdkv_results"
os.makedirs(OUT, exist_ok=True)
res = REPO + "/results/longbench"
if not os.path.islink(res):
    for f in glob.glob(res + "/*"):
        if not os.path.exists(OUT + "/" + os.path.basename(f)):
            shutil.copy(f, OUT)
    shutil.rmtree(res)
    os.symlink(OUT, res)
os.chdir(REPO)
print("repo ready; results ->", OUT, "| files there:", sorted(os.listdir(OUT)))

In [ ]:
# 3. Install
!pip install -q "kvpress==0.5.5" rouge rapidfuzz datasets peft pytest requests
if subprocess.run([sys.executable, "-c", "import fire"], capture_output=True).returncode != 0:
    !pip install -q -U "fire>=0.7"
!python -c "import torch, transformers, kvpress; print('torch', torch.__version__, '| transformers', transformers.__version__, '| kvpress OK |', torch.cuda.get_device_name(0))"

In [ ]:
# 4. Unit tests (expect: 9 passed)
!python -m pytest -q members/manjot/test_rdkv.py

In [ ]:
# 5. Build the 800-record dev set and check it matches the baseline
import json
DATA = "data/processed_longbench/processed_longbench_dev.jsonl"
if not os.path.exists(DATA) or sum(1 for _ in open(DATA)) < 800:
    if os.path.exists(DATA): os.remove(DATA)
    !python members/samruddhi/build_dev_data.py --token $HF_TOKEN
built = {json.loads(l)["record_id"]: json.loads(l)["prompt_tokens_after"] for l in open(DATA)}
base = {json.loads(l)["record_id"]: json.loads(l)["prompt_tokens"] for l in open("results/longbench/Llama-3.1-8B-Instruct_none_0.0.jsonl")}
print(f"{len(built)} records; identical to the baseline for {sum(built.get(k) == v for k, v in base.items())}/{len(base)}")

In [ ]:
# 6. FULL RUNS (bf16, one GPU = the baseline's setup). Resumes automatically after a disconnect.
for r in RATIOS:
    !python harness/run_longbench.py --press rdkv --compression-ratio {r}
    !python members/manjot/compare.py results/longbench/Llama-3.1-8B-Instruct_rdkv_{r}.jsonl